In [2]:
import sympy as sp
import sympy.diffgeom as dg

In [14]:
state_mfld = dg.Manifold("M", 5)
state_mfld_patch = dg.Patch("P", state_mfld)

x, y, theta, v, omega = sp.symbols(r"x,y,\theta,v,\omega", real=True)

state_mfld_coords = dg.CoordSystem(
    "StateSpace", state_mfld_patch, (x, y, theta, v, omega)
)

(x_sc, y_sc, theta_sc, v_sc, omega_sc) = state_mfld_coords.base_scalars()
(x_vec, y_vec, theta_vec, v_vec, omega_vec) = state_mfld_coords.base_vectors()

In [17]:
# define the vector fields of the system

f = (
    v_sc * sp.cos(theta_sc) * x_vec
    + v_sc * sp.sin(theta_sc) * y_vec
    + omega_sc * theta_vec
)

f1 = v_vec
f2 = omega_vec

display(f, f1, f2)

sin(\theta)*v*e_y + cos(\theta)*v*e_x + \omega*e_\theta

e_v

e_\omega

In [ ]:
# define the variables associated with tthe trajectory

t = sp.symbols("t", real=True)
alpha, beta = sp.symbols(r"\alpha,\beta", real=True)

x_tr = sp.Function("x_tr")(t)
y_tr = sp.Function("y_tr")(t)
theta_tr = sp.Function(r"\theta_tr")(t)

# define the constraint to enforce decreasing error
v_fn = (
    0.5 * alpha * ((x_sc - x_tr) ** 2 + (y_sc - y_tr) ** 2)
    + 0.5 * beta * (theta_sc - theta_tr) ** 2
)
display(v_fn)

0.5*\alpha*((-x_tr(t) + x)**2 + (-y_tr(t) + y)**2) + 0.5*\beta*(-\theta_tr(t) + \theta)**2

In [47]:
Lf_Lf_v = dg.LieDerivative(f, dg.LieDerivative(f, v_fn)).nsimplify().expand()
Lf1_Lf_v = dg.LieDerivative(f1, dg.LieDerivative(f, v_fn)).nsimplify().expand()
Lf2_Lf_v = dg.LieDerivative(f2, dg.LieDerivative(f, v_fn)).nsimplify().expand()
dv_dt = sp.diff(v_fn, t).nsimplify().expand()

display(Lf_Lf_v)
display(Lf1_Lf_v)
display(Lf2_Lf_v)
display(dv_dt)

\alpha*x_tr(t)*sin(\theta)*v*\omega - \alpha*y_tr(t)*cos(\theta)*v*\omega + \alpha*sin(\theta)**2*v**2 - \alpha*sin(\theta)*x*v*\omega + \alpha*cos(\theta)**2*v**2 + \alpha*cos(\theta)*y*v*\omega + \beta*\omega**2

-\alpha*x_tr(t)*cos(\theta) - \alpha*y_tr(t)*sin(\theta) + \alpha*sin(\theta)*y + \alpha*cos(\theta)*x

-\beta*\theta_tr(t) + \beta*\theta

\alpha*x_tr(t)*Derivative(x_tr(t), t) + \alpha*y_tr(t)*Derivative(y_tr(t), t) - \alpha*x*Derivative(x_tr(t), t) - \alpha*y*Derivative(y_tr(t), t) + \beta*\theta_tr(t)*Derivative(\theta_tr(t), t) - \beta*\theta*Derivative(\theta_tr(t), t)

In [45]:
# define the inputs
u_f, u_t = sp.symbols("F,T", real=True)

# define the optimization problem

p, delta = sp.symbols(r"p,delta", real=True)
prob_f = (0.5 * (u_f**2 + u_t**2) + p * delta**2).simplify()
prob_g = (Lf_Lf_v + Lf1_Lf_v * u_f + Lf2_Lf_v * u_t + dv_dt <= delta).simplify()
prob_vars = [u_f, u_t, delta]

display(prob_f)
display(prob_g)

0.5*F**2 + 0.5*T**2 + delta**2*p

delta >= -F*\alpha*((x_tr(t) - x)*cos(\theta) + (y_tr(t) - y)*sin(\theta)) - T*\beta*(\theta_tr(t) - \theta) + \alpha*((x_tr(t) - x)*Derivative(x_tr(t), t) + (y_tr(t) - y)*Derivative(y_tr(t), t)) + \alpha*sin(\theta)**2*v**2 + \alpha*cos(\theta)**2*v**2 + \beta*(\theta_tr(t) - \theta)*Derivative(\theta_tr(t), t) + (\alpha*(x_tr(t) - x)*sin(\theta)*v - \alpha*(y_tr(t) - y)*cos(\theta)*v + \beta*\omega)*\omega